# 01 — Qualidade dos dados e análise exploratória

Este notebook inicia a **Análise Exploratória de Dados (EDA)** do painel municipal português.

Objetivos:

1. carregar o ficheiro processado sem alterar os dados originais;
2. confirmar a estrutura e a chave `município + ano`;
3. estudar os valores em falta;
4. resumir as distribuições das variáveis;
5. observar a evolução temporal;
6. explorar relações iniciais entre habitação e demografia.

> Uma associação ou correlação entre variáveis não demonstra causalidade.


## 1. Importações e localização do projeto

`pandas` trata os dados tabulares e `matplotlib` cria os gráficos. A função abaixo encontra a raiz do projeto, quer o notebook seja iniciado na pasta principal ou dentro de `notebooks/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Não foi possível encontrar a raiz do projeto.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_PATH = PROJECT_ROOT / "data/processed/municipality_year_panel_2021_2025.csv"
DATA_PATH


## 2. Carregamento dos dados

O código municipal é lido como texto porque é um identificador, não uma quantidade para calcular. As células vazias das variáveis numéricas são convertidas pelo `pandas` para `NaN`, que significa valor em falta.


In [ ]:
panel = pd.read_csv(
    DATA_PATH,
    dtype={"municipality_code": "string"},
)

print(f"Linhas: {panel.shape[0]:,}")
print(f"Colunas: {panel.shape[1]}")
panel.head()


## 3. Significado das colunas

| Coluna | Significado | Unidade |
|---|---|---|
| `municipality_code` | Código oficial do município | identificador |
| `municipality_name` | Nome do município | texto |
| `year` | Ano da observação | ano |
| `housing_price_m2_total` | Valor mediano de venda dos alojamentos familiares | €/m² |
| `population_total` | População residente | pessoas |
| `aging_index` | Pessoas com 65+ anos por cada 100 jovens dos 0–14 | razão |
| `migration_balance` | Diferença entre entradas e saídas migratórias | pessoas |
| `income_after_tax_per_tax_household` | Rendimento declarado menos IRS liquidado por agregado fiscal | euros |

O rendimento não representa salário médio. O preço `Total` também não é a soma das categorias `Novos` e `Existentes`.


## 4. Estrutura e tipos

Nesta etapa verificamos como o `pandas` interpretou cada coluna. As cinco medições devem ser numéricas; os códigos e nomes identificam o município.


In [ ]:
panel.info()


## 5. Testes rápidos de integridade

Esperamos 308 municípios durante cinco anos, ou seja, 1 540 linhas. Cada combinação de município e ano deve aparecer apenas uma vez. Estas verificações param o notebook se a base não respeitar essas regras.


In [ ]:
key = ["municipality_code", "year"]

assert len(panel) == 308 * 5, "O painel não tem as 1 540 linhas esperadas."
assert panel["municipality_code"].nunique() == 308, "O número de municípios mudou."
assert sorted(panel["year"].unique().tolist()) == [2021, 2022, 2023, 2024, 2025]
assert not panel.duplicated(key).any(), "Existem chaves município–ano repetidas."

print("✓ Estrutura esperada: 308 municípios × 5 anos")
print("✓ Nenhuma chave município–ano repetida")


## 6. Valores em falta

Primeiro contamos as ausências em cada coluna. A percentagem usa as 1 540 linhas como denominador.


In [ ]:
missing_summary = pd.DataFrame(
    {
        "missing": panel.isna().sum(),
        "percent": panel.isna().mean().mul(100),
    }
).sort_values("missing", ascending=False)

missing_summary


Agora dividimos as ausências por ano. Isto permite distinguir dois padrões:

- preços de habitação ausentes em alguns municípios;
- rendimento ausente em todos os municípios de 2025 porque a fonte termina em 2024.


In [ ]:
measurement_columns = [
    "housing_price_m2_total",
    "population_total",
    "aging_index",
    "migration_balance",
    "income_after_tax_per_tax_household",
]

missing_by_year = panel.groupby("year")[measurement_columns].agg(
    lambda column: column.isna().sum()
)
missing_by_year


In [ ]:
ax = missing_by_year.plot(
    kind="bar",
    figsize=(11, 5),
    title="Valores em falta por variável e ano",
)
ax.set_xlabel("Ano")
ax.set_ylabel("Número de municípios")
ax.legend(title="Variável", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


### Conclusão — cobertura dos dados

O painel está estruturalmente completo: contém os 308 municípios em todos os anos. As células vazias são **medições não disponíveis**, não municípios que desapareceram da base.

- O preço da habitação tem 33 valores ausentes em 1 540 linhas (2,1%).
- O rendimento tem 308 valores ausentes, todos em 2025, porque a fonte termina em 2024.
- População, índice de envelhecimento e saldo migratório não têm valores ausentes.

Consequentemente, análises com rendimento usarão 2021–2024. Os valores ausentes do preço serão excluídos apenas dos cálculos que precisem dessa variável e nunca serão convertidos em zero.


## 7. Estatísticas descritivas

Estas medidas ajudam a detetar escalas, dispersão e possíveis valores extremos. A mediana é o percentil `50%`.


In [ ]:
panel[measurement_columns].describe().T


### Conclusão — escala e dispersão

Os municípios são muito diferentes em dimensão. A população mediana é 14 128,5 residentes, enquanto a média é 36 067,8 e o máximo ultrapassa 665 mil. Esta diferença mostra a influência dos municípios de grande dimensão.

O preço também apresenta valores elevados numa parte pequena da distribuição: a mediana conjunta é 845 €/m², a média é 1 055,24 €/m² e o máximo é 4 875 €/m². O saldo migratório varia entre -5 886 e 58 869 pessoas, pelo que os valores absolutos favorecem municípios maiores. Na análise seguinte devemos calcular o saldo migratório por mil habitantes.


## 8. Evolução anual

Calculamos a mediana entre municípios em cada ano. Esta é uma descrição do município mediano da base; não corresponde necessariamente a uma estatística oficial agregada para Portugal.


In [ ]:
yearly_medians = panel.groupby("year")[measurement_columns].median()
yearly_medians


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
series_to_plot = [
    ("housing_price_m2_total", "Preço mediano da habitação", "€/m²"),
    ("population_total", "População municipal mediana", "Pessoas"),
    ("aging_index", "Índice de envelhecimento mediano", "Índice"),
    ("migration_balance", "Saldo migratório mediano", "Pessoas"),
]

for ax, (column, title, ylabel) in zip(axes.flat, series_to_plot):
    yearly_medians[column].plot(ax=ax, marker="o")
    ax.set_title(title)
    ax.set_xlabel("Ano")
    ax.set_ylabel(ylabel)

plt.tight_layout()
plt.show()


### Conclusão — evolução temporal

Entre 2021 e 2025, a mediana dos preços municipais passou de 728,5 para 1 108 €/m², um aumento de aproximadamente **52,1%**. No mesmo cálculo, a população municipal mediana aumentou cerca de 3,9% e o índice de envelhecimento mediano cerca de 5,2%.

O rendimento mediano passou de 15 124,5 € em 2021 para 18 313,5 € em 2024, um aumento de aproximadamente 21,1%. Não podemos calcular 2025 porque esse ano ainda não está disponível. Estes valores são nominais e ainda não foram corrigidos pela inflação.

Estas medianas resumem os 308 municípios. Não mostram se todos evoluíram da mesma maneira, pelo que o passo seguinte será calcular variações para cada município.


## 9. Distribuição dos preços

O histograma mostra quantos municípios aparecem em cada intervalo de preço. Uma cauda longa à direita significa que existem alguns municípios com valores muito superiores à maioria.


In [ ]:
ax = panel["housing_price_m2_total"].plot(
    kind="hist",
    bins=30,
    figsize=(10, 5),
    edgecolor="white",
    title="Distribuição dos preços municipais, 2021–2025",
)
ax.set_xlabel("Preço mediano de venda (€/m²)")
ax.set_ylabel("Número de observações município–ano")
plt.tight_layout()
plt.show()


### Conclusão — distribuição dos preços

A distribuição tem uma cauda para a direita: a maioria das observações apresenta preços relativamente baixos e existe um grupo menor com preços muito elevados. A média superior à mediana confirma essa assimetria. Cerca de 90% das observações município–ano ficam até aproximadamente 2 047 €/m².

Por causa desta assimetria, medianas e percentis descrevem melhor o município típico do que apenas a média. Em modelos futuros também poderá ser útil testar o logaritmo do preço.


## 10. Relação entre rendimento e preço em 2024

Usamos 2024 porque é o ano mais recente com rendimento disponível. Cada ponto representa um município. Municípios sem preço divulgado ficam automaticamente fora deste gráfico.


In [ ]:
panel_2024 = panel.loc[
    panel["year"].eq(2024),
    [
        "municipality_name",
        "housing_price_m2_total",
        "income_after_tax_per_tax_household",
    ],
].dropna()

ax = panel_2024.plot.scatter(
    x="income_after_tax_per_tax_household",
    y="housing_price_m2_total",
    figsize=(9, 6),
    alpha=0.65,
    title="Rendimento fiscal e preço da habitação — 2024",
)
ax.set_xlabel("Rendimento por agregado fiscal (€)")
ax.set_ylabel("Preço mediano de venda (€/m²)")
plt.tight_layout()
plt.show()


In [ ]:
price_income_correlation_2024 = panel_2024[
    ["housing_price_m2_total", "income_after_tax_per_tax_household"]
].corr().iloc[0, 1]

print(f"Municípios utilizados: {len(panel_2024)}")
print(f"Correlação preço–rendimento em 2024: {price_income_correlation_2024:.3f}")


### Conclusão — rendimento e preço em 2024

Existem 303 municípios com os dois valores disponíveis em 2024. A correlação linear entre rendimento fiscal por agregado e preço da habitação é aproximadamente **0,539**, uma associação positiva moderada.

Isto indica que municípios com maior rendimento tendem a apresentar preços mais elevados, mas o gráfico não demonstra causalidade nem mede sozinho a acessibilidade à habitação. Para comparar pressão habitacional, criaremos um indicador explícito que relacione preço por m² e rendimento anual, identificado como uma aproximação e não como uma taxa oficial de esforço.


## 11. Correlações iniciais

A correlação varia entre -1 e 1. Valores próximos de 1 indicam que duas variáveis tendem a aumentar em conjunto; valores próximos de -1 indicam movimentos opostos. A correlação ignora automaticamente pares com valores em falta e não demonstra causalidade.


In [ ]:
correlations = panel[measurement_columns].corr()
correlations


In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
image = ax.imshow(correlations, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(correlations.columns)), correlations.columns, rotation=45, ha="right")
ax.set_yticks(range(len(correlations.index)), correlations.index)

for row in range(len(correlations.index)):
    for column in range(len(correlations.columns)):
        ax.text(column, row, f"{correlations.iloc[row, column]:.2f}", ha="center", va="center")

fig.colorbar(image, ax=ax, label="Correlação")
ax.set_title("Matriz de correlações — observações disponíveis")
plt.tight_layout()
plt.show()


### Conclusão — relações iniciais

No conjunto de 2021–2025, o preço apresenta correlação positiva com população (0,619), rendimento (0,548) e saldo migratório (0,419), e correlação negativa com o índice de envelhecimento (-0,542).

Estas correlações misturam diferenças entre municípios e mudanças ao longo do tempo. Também usam valores absolutos de população e migração. Por isso, servem para escolher perguntas, mas ainda não sustentam conclusões explicativas. A análise seguinte deve trabalhar com taxas, variações anuais e separação por ano.


## 12. Municípios com preços mais altos em 2025

Esta tabela é descritiva. Não explica por que razão os preços são elevados e não deve ser interpretada como uma relação causal.


In [ ]:
(
    panel.loc[panel["year"].eq(2025)]
    .nlargest(10, "housing_price_m2_total")
    [["municipality_name", "housing_price_m2_total", "population_total", "migration_balance"]]
    .reset_index(drop=True)
)


### Conclusão — preços mais elevados em 2025

Lisboa apresenta o maior preço observado em 2025, seguida por Cascais, Oeiras, Loulé e Lagos. Entre os valores disponíveis, o intervalo vai de 245 €/m² em Sernancelhe a 4 875 €/m² em Lisboa.

A amplitude confirma que uma média nacional esconderia diferenças municipais muito grandes. Para explicar estes contrastes será necessário comparar evolução, rendimento e dinâmica demográfica, e mais tarde acrescentar uma classificação regional.


## Direção da investigação

Com base nesta EDA, o projeto seguirá três perguntas principais:

| Pergunta | Medida principal | Período | Cuidado de interpretação |
|---|---|---|---|
| **1. Como evoluíram os preços da habitação em cada município?** | Variação percentual do preço por m² e diferença entre 2021 e 2025 | 2021–2025 | Municípios sem preço num dos extremos não podem entrar nessa comparação |
| **2. Que relação existe entre rendimento fiscal e preço da habitação?** | Correlação por ano e indicador exploratório `preço por m² / rendimento anual` | 2021–2024 | O indicador é uma aproximação, não uma taxa oficial de esforço nem prova causalidade |
| **3. A dinâmica migratória está associada ao crescimento dos preços?** | Saldo migratório por mil habitantes e crescimento anual do preço | 2021–2025 | Devemos controlar o ano e a dimensão do município; associação não significa causalidade |

O índice de envelhecimento e a evolução populacional serão usados como variáveis de contexto. Antes de responder às perguntas, criaremos em `src/features/` as taxas e variações necessárias, com testes para confirmar os cálculos.
